# Model Selection and Hyperparameter Tuning

**Use scikit-learn as much as possible!**

In [4]:
import pandas as pd
import numpy as np

## Ames Housing Data

As usual, we'll try to predict SalePrice using a mix of quantitative and categorical features. See the [data documentation](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt) for all the variables that are available in the data set. We'll try to come up with a model that minimizes test error, measured by root mean square error (RMSE).

In [5]:
df_ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


1\. Recall that in a previous notebook we fit a 10-nearest neighbors regression model that predicts the price (just **SalePrice**, not log) of a home using square footage (**Gr Liv Area**), number of bedrooms (**Bedroom AbvGr**), number of full bathrooms (**Full Bath**), number of half bathrooms (**Half Bath**), and **Neighborhood**. But what about other sets of features?

Specify 5 sets of features to investigate. These sets could be subsets of the variables above, but you should also include other variables from the data set. The choice is up to you, but try to guess which sets might lead to good models based on your intuition and what we have already seen for this data set.

For each set of features, fit a 10-nearest neighbors regression model and estimate its test error (RMSE) using cross-validation. You can write a loop to iterate over the sets of features similar to what was done in the Model Selection section of the pre-class notebook.

Report your lowest RMSE and the corresponding features.

In [6]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# Drop rows with missing values in our candidate features to keep things clean
candidate_cols = [
    "Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood",
    "Overall Qual", "Year Built", "Garage Area", "Total Bsmt SF", "Lot Area"
]

# Clean target and features of missing data for comparison
df_clean = df_ames.dropna(subset=candidate_cols + ["SalePrice"]).copy()
X_all = df_clean[candidate_cols]
y = df_clean["SalePrice"]

# Define 5 sets of features to investigate
feature_sets = {
    "Set 1 (Original)": ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"],
    "Set 2 (Set 1 + Overall Quality & Year Built)": ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood", "Overall Qual", "Year Built"],
    "Set 3 (Set 2 + Garage & Basement Areas)": ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood", "Overall Qual", "Year Built", "Garage Area", "Total Bsmt SF"],
    "Set 4 (Only Size & Quality related)": ["Gr Liv Area", "Overall Qual", "Garage Area", "Total Bsmt SF"],
    "Set 5 (Set 3 + Lot Area)": ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood", "Overall Qual", "Year Built", "Garage Area", "Total Bsmt SF", "Lot Area"]
}

results = {}

# Iterate over each feature set
for name, features in feature_sets.items():
    # Identify which features are categorical and which are numeric in this set
    cat_features = [col for col in features if df_clean[col].dtype == "object"]
    num_features = [col for col in features if df_clean[col].dtype != "object"]

    # Preprocessor: scale numeric columns and one-hot encode categorical columns
    preprocessor = make_column_transformer(
        (StandardScaler(), num_features),
        (OneHotEncoder(handle_unknown="ignore"), cat_features),
        remainder="drop"
    )

    # 10-NN pipeline
    pipeline = make_pipeline(
        preprocessor,
        KNeighborsRegressor(n_neighbors=10)
    )

    # Perform 5-fold cross-validation using Negative MSE
    scores = cross_val_score(pipeline, X_all[features], y, cv=5, scoring="neg_mean_squared_error")

    # Convert MSE to RMSE
    rmse_scores = np.sqrt(-scores)
    mean_rmse = rmse_scores.mean()
    results[name] = mean_rmse
    print(f"{name} -> Mean RMSE: {mean_rmse:.2f}")

# Find the best feature set
best_set = min(results, key=results.get)
print(f"\nLowest RMSE: {results[best_set]:.2f} using {best_set}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 856, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.14/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/lib/python3.14/site-packages/sklearn/pipeline.py", line 636, in fit
    Xt = self._fit(
        X, y, routed_params, raw_params=params, callback_ctx=callback_ctx
    )
  File "/opt/anaconda3/lib/python3.14/site-packages/sklearn/pipeline.py", line 565, in _fit
   

2\. Recall that in a previous notebook we fit a $k$-nearest neighbors regression model that predicts the price (just **SalePrice**, not log) of a home using square footage (**Gr Liv Area**), number of bedrooms (**Bedroom AbvGr**), number of full bathrooms (**Full Bath**), number of half bathrooms (**Half Bath**), and **Neighborhood**. Previously we used a loop to iterate and find the value of $k$ with the smallest test error (RMSE).

Now use grid search in scikit-learn to implement this process. Use the results to create a plot of test error versus $k$.

In [ ]:
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

# We will use the original feature set (Set 1)
features_q2 = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"]

# Identify categorical and numeric columns
cat_features_q2 = [col for col in features_q2 if df_clean[col].dtype == "object"]
num_features_q2 = [col for col in features_q2 if df_clean[col].dtype != "object"]

# Set up preprocessor
preprocessor_q2 = make_column_transformer(
    (StandardScaler(), num_features_q2),
    (OneHotEncoder(handle_unknown="ignore"), cat_features_q2),
    remainder="drop"
)

# Create a pipeline with a placeholder KNeighborsRegressor
pipeline_q2 = make_pipeline(
    preprocessor_q2,
    KNeighborsRegressor()
)

# Define parameter grid. Notice the naming convention for pipeline steps:
# 'kneighborsregressor__n_neighbors'
param_grid = {
    'kneighborsregressor__n_neighbors': list(range(1, 51))
}

# Setup GridSearchCV
grid_search = GridSearchCV(
    pipeline_q2,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# Fit the grid search
grid_search.fit(X_all[features_q2], y)

# Extract grid search results
k_values = param_grid['kneighborsregressor__n_neighbors']
mean_mse_scores = grid_search.cv_results_['mean_test_score']
mean_rmse_scores = np.sqrt(-mean_mse_scores)

# Display best parameters
print(f"Best k: {grid_search.best_params_['kneighborsregressor__n_neighbors']}")
print(f"Lowest RMSE: {np.sqrt(-grid_search.best_score_):.2f}")

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(k_values, mean_rmse_scores, marker='o', linestyle='-')
plt.title("Cross-Validation RMSE vs. k (n_neighbors)")
plt.xlabel("k (Number of Neighbors)")
plt.ylabel("Mean RMSE (5-Fold CV)")
plt.grid(True)
plt.show()

In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

3\. $k$-nearest neighbors relies on distance and we know there are different distance metrics. Repeat the previous part and use grid search to iterate over both (1) `n_neighbors` and (2) `metric`, which can be `"euclidean"` or `"manhattan"` (see the the [`KNeighborsRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)). Compare the results to the previous part. (Hint: recall the note in the pre-class notebook about grid search conventions for refering to steps in the pipeline.)

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

# Re-define original features and pipeline to make this cell self-contained
features_q2 = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"]

cat_features_q2 = [col for col in features_q2 if df_clean[col].dtype == "object"]
num_features_q2 = [col for col in features_q2 if df_clean[col].dtype != "object"]

preprocessor_q2 = make_column_transformer(
    (StandardScaler(), num_features_q2),
    (OneHotEncoder(handle_unknown="ignore"), cat_features_q2),
    remainder="drop"
)

pipeline_q2 = make_pipeline(
    preprocessor_q2,
    KNeighborsRegressor()
)

# Define parameter grid for both n_neighbors and metric
param_grid_q3 = {
    'kneighborsregressor__n_neighbors': list(range(1, 51)),
    'kneighborsregressor__metric': ['euclidean', 'manhattan']
}

# Setup GridSearchCV with the new parameter grid
grid_search_q3 = GridSearchCV(
    pipeline_q2,
    param_grid=param_grid_q3,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# Fit the grid search
grid_search_q3.fit(X_all[features_q2], y)

# Get the best parameters and lowest RMSE
best_params = grid_search_q3.best_params_
best_rmse = np.sqrt(-grid_search_q3.best_score_)

print(f"Best Parameters: {best_params}")
print(f"Lowest RMSE: {best_rmse:.2f}")

Best Parameters: {'kneighborsregressor__metric': 'manhattan', 'kneighborsregressor__n_neighbors': 5}
Lowest RMSE: 38632.63


4\. Combine elements of the previous parts and fit what you think will be the best $k$-nearest neighbors model. Specify which features you are using, the value of $k$, and the distance metric. Estimate the test error. (In 2-3 you used the specified set of features. Now you can combine elements of 2-3 with the features you chose in part 1, or other sets of features, as well as investigating other parameter choices.)

Note: Look at the [`KNeighborsRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) for a full list of parameters. You might consider tuning `n_neighbors`, `metric` (or `p`), and/or `weights`. You could also try other elements like scaling too.

In [ ]:

best_features = [
    "Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood",
    "Overall Qual", "Year Built", "Garage Area", "Total Bsmt SF", "Lot Area"
]


cat_features_best = [col for col in best_features if df_clean[col].dtype == "object"]
num_features_best = [col for col in best_features if df_clean[col].dtype != "object"]


preprocessor_best = make_column_transformer(
    (StandardScaler(), num_features_best),
    (OneHotEncoder(handle_unknown="ignore"), cat_features_best),
    remainder="drop"
)


pipeline_best = make_pipeline(
    preprocessor_best,
    KNeighborsRegressor()
)


param_grid_best = {
    'kneighborsregressor__n_neighbors': list(range(1, 31)),
    'kneighborsregressor__metric': ['euclidean', 'manhattan'],
    'kneighborsregressor__weights': ['uniform', 'distance']
}


grid_search_best = GridSearchCV(
    pipeline_best,
    param_grid=param_grid_best,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search_best.fit(X_all[best_features], y)

#  results
best_params_best = grid_search_best.best_params_
best_rmse_best = np.sqrt(-grid_search_best.best_score_)

print(f"Best Hyperparameters: {best_params_best}")
print(f"Estimated Test Error (RMSE): {best_rmse_best:.2f}")

Best Hyperparameters: {'kneighborsregressor__metric': 'manhattan', 'kneighborsregressor__n_neighbors': 6, 'kneighborsregressor__weights': 'distance'}
Estimated Test Error (RMSE): 29882.89


5\. Now fit a linear regression model using the same features that you used in the previous part. Estimate the test error. How does the test error for linear regression compare to the kNN model?


In [ ]:
from sklearn.linear_model import LinearRegression

# Create a pipeline with the same preprocessor but with LinearRegression
pipeline_lr = make_pipeline(
    preprocessor_best,
    LinearRegression()
)

# Calculate cross-validation scores for Linear Regression
scores_lr = cross_val_score(
    pipeline_lr,
    X_all[best_features],
    y,
    cv=5,
    scoring="neg_mean_squared_error"
)

# Calculate mean RMSE
rmse_lr = np.sqrt(-scores_lr).mean()

print(f"Linear Regression Mean RMSE: {rmse_lr:.2f}")
print(f"Best kNN Mean RMSE (from Q4): {best_rmse_best:.2f}")
print(f"Difference: {rmse_lr - best_rmse_best:.2f}")

Linear Regression Mean RMSE: 33639.76
Best kNN Mean RMSE (from Q4): 29882.89
Difference: 3756.87


## Loans Data

The loans data set represents thousands of loans made through the Lending Club platform, which is a platform that allows individuals to lend to other individuals. Of course, not all loans are created equal. Someone who is a essentially a sure bet to pay back a loan will have an easier time getting a loan with a low interest rate than someone who appears to be riskier. And for people who are very risky? They may not even get a loan offer, or they may not have accepted the loan offer due to a high interest rate. It is important to keep that last part in mind, since this dataset only represents loans actually made, i.e. do not mistake this data for loan applications!

[Some information about the data set is here](https://openintrostat.github.io/openintro/reference/loans_full_schema.html). There are also some related exercises in this [OpenIntro textbook](https://openintro-ims.netlify.app/inf-model-mlr.html).

**Our goal will be to develop a regression model for predicting `interest_rate`.**

We will have a competition to see who can come up with the model with the smallest test RMSE. Fit several models and try to find the smallest test RMSE. When you think you have a good one, share your model and its test RMSE. Then see what other people have done; can you do even better?

Note: when you are consider features to include, you might want to consider transformations of the variables.

You should use `GridSearchCV` to tune at least one parameter.

Note: you should do some exploration of the data first before you start fitting models.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/refs/heads/main/data/loans_full_schema.csv")

df

,Unnamed: 0,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,1,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,2,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,3,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,4,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,5,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,owner,10.0,TX,RENT,108000.0,Source Verified,22.28,NaN,NaN,...,A4,Jan-2018,Current,whole,Cash,21586.34,2969.80,2413.66,556.14,0.0
9996,9997,director,8.0,PA,MORTGAGE,121000.0,Verified,32.38,NaN,NaN,...,D3,Feb-2018,Current,whole,Cash,9147.44,1456.31,852.56,603.75,0.0
9997,9998,toolmaker,10.0,CT,MORTGAGE,67000.0,Verified,45.26,107000.0,Source Verified,...,E2,Feb-2018,Current,fractional,Cash,27617.65,4620.80,2382.35,2238.45,0.0
9998,9999,manager,1.0,WI,MORTGAGE,80000.0,Source Verified,11.99,NaN,NaN,...,A1,Feb-2018,Current,whole,Cash,21518.12,2873.31,2481.88,391.43,0.0


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

# Let's inspect columns first or pick strong intuitive predictors:
# 'loan_amount', 'annual_income', 'debt_to_income', 'credit_util', 'homeownership', 'verified_income'
predictors = ['loan_amount', 'annual_income', 'debt_to_income', 'homeownership', 'verified_income']

# Drop rows with missing values in our predictors or target for baseline evaluation
df_loans_clean = df.dropna(subset=predictors + ['interest_rate']).copy()
X_loans = df_loans_clean[predictors]
y_loans = df_loans_clean['interest_rate']

# Set up preprocessor
cat_cols = ['homeownership', 'verified_income']
num_cols = ['loan_amount', 'annual_income', 'debt_to_income']

preprocessor_loans = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(handle_unknown='ignore'), cat_cols),
    remainder='drop'
)

# Create pipeline
pipeline_loans = make_pipeline(
    preprocessor_loans,
    KNeighborsRegressor()
)

# Define hyperparameter grid to search over
param_grid_loans = {
    'kneighborsregressor__n_neighbors': list(range(1, 31)),
    'kneighborsregressor__weights': ['uniform', 'distance'],
    'kneighborsregressor__metric': ['euclidean', 'manhattan']
}

# Run GridSearchCV
grid_loans = GridSearchCV(
    pipeline_loans,
    param_grid=param_grid_loans,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_loans.fit(X_loans, y_loans)

best_rmse_loans = np.sqrt(-grid_loans.best_score_)
print(f"Best Hyperparameters: {grid_loans.best_params_}")
print(f"Best CV RMSE: {best_rmse_loans:.4f}")

Best Hyperparameters: {'kneighborsregressor__metric': 'manhattan', 'kneighborsregressor__n_neighbors': 30, 'kneighborsregressor__weights': 'uniform'}
Best CV RMSE: 4.7130
